In [5]:
import numpy as np
import pandas as pd

df = pd.read_csv('../data/processed/hourly_demand_weather.csv', parse_dates=['date'])

TARGET = 'trips'
TIME = ['hour_sin', 'hour_cos', 'is_weekend', 'is_holiday']
LAGS = ['lag_24', 'lag_168', 'roll_24']
WEATHER = ['temperature_2m', 'precipitation', 'wind_speed_10m', 'relative_humidity_2m', 'is_rain']

use = df[df['exclude'] == 0]
train = use[use['split'] == 'train']
val = use[use['split'] == 'val']
test = use[use['split'] == 'test']
print(len(train), len(val), len(test))

10058 2232 2304


## 2. Evaluation function

In [6]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate(y_true, y_pred): # all models are compared with the same metrics (MAE, RMSE, R²). Predictions are clipped at zero because trips cannot be negative
    y_pred = np.clip(y_pred, 0, None)
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred),
    }

## 3. Model 0: baseline (same hour last week)

In [7]:
for name, part in [('val', val), ('test', test)]: # The simplest forecast sets the minimum accuracy that all models must beat
    print(name, evaluate(part[TARGET], part['lag_168']))

val {'MAE': 8.376792114695341, 'RMSE': np.float64(13.5033349126592), 'R2': 0.6166396580855378}
test {'MAE': 7.081597222222222, 'RMSE': np.float64(11.899244257047961), 'R2': 0.766407353231628}


## 4. Model 1: Ridge regression
Why: simple, interpretable linear model. City is one-hot encoded and numeric features are scaled inside the pipeline, fitted on the training set only. Feature groups are added step by step (time, lags, weather).

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

def make_ridge(num_cols, alpha):
    prep = ColumnTransformer([
        ('city', OneHotEncoder(handle_unknown='ignore'), ['city']),
        ('num', StandardScaler(), num_cols),
    ])
    return Pipeline([('prep', prep), ('model', Ridge(alpha=alpha))])

In [10]:
sets = {'time': TIME, 'time+lags': TIME + LAGS, 'time+lags+weather': TIME + LAGS + WEATHER}
rows = []
for name, cols in sets.items():
    for a in [0.1, 1, 10, 100]:
        m = make_ridge(cols, a).fit(train[['city'] + cols], train[TARGET])
        res = evaluate(val[TARGET], m.predict(val[['city'] + cols]))
        rows.append({'model': 'Ridge', 'features': name, 'alpha': a, **res})

ridge_val = pd.DataFrame(rows).round(2)
ridge_val

,model,features,alpha,MAE,RMSE,R2
0,Ridge,time,0.1,13.69,18.96,0.24
1,Ridge,time,1.0,13.69,18.95,0.24
2,Ridge,time,10.0,13.67,18.91,0.25
3,Ridge,time,100.0,13.53,18.51,0.28
4,Ridge,time+lags,0.1,6.53,10.06,0.79
5,Ridge,time+lags,1.0,6.53,10.06,0.79
6,Ridge,time+lags,10.0,6.53,10.07,0.79
7,Ridge,time+lags,100.0,6.53,10.09,0.79
8,Ridge,time+lags+weather,0.1,6.89,10.19,0.78
9,Ridge,time+lags+weather,1.0,6.89,10.19,0.78
